In [2]:

# ── Imports ────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os


In [1]:

# ── Constants ──────────────────────────────────────────────────────────────────
ROOT = "../../"

# Input paths
PARCEL_SHP      = ROOT + "data/geo/Parcels_7711352183771741448/Parcel_Boundaries.shp"
BUSINESS_XLSX   = ROOT + "data/business/Oakland_DataAxleV2.xlsx"

# Output paths
OUT_DIR         = ROOT + "data/corridors/parcels/"
OUT_USE_CODE    = OUT_DIR + "parcels_use_code_filtered.gpkg"
OUT_BIZ_CONTAIN = OUT_DIR + "parcels_with_business.gpkg"
OUT_UNION       = OUT_DIR + "parcels_union.gpkg"

# Use code prefixes to keep (3xxx, 8xxx, 9xxx)
USE_CODE_PREFIXES = ("3", "8", "9")

# Coordinate reference systems
PARCEL_CRS  = "EPSG:3857"   # Web Mercator (native CRS of the parcel shapefile)
BUSINESS_CRS = "EPSG:4326"  # WGS 84 (lat/lon of the business data)



# Section 1 — Identify Preliminary Corridor Parcels

This section filters the Alameda County parcel dataset to produce three targeted parcel layers,
all saved as GeoPackage files under `data/corridors/parcels/`:

| Output file | Description |
|---|---|
| `parcels_use_code_filtered.gpkg` | Parcels whose `UseCode` starts with 3, 8, or 9 (commercial / industrial / special use) |
| `parcels_with_business.gpkg` | Parcels that contain at least one DataAxle business centroid |
| `parcels_union.gpkg` | Union of the two filtered sets (deduplicated) |



## 1.1 Load Parcel Data

Read the full Alameda County parcel shapefile into a GeoDataFrame.  
The native CRS is **EPSG:3857** (Web Mercator).


In [3]:

# Load full parcel shapefile (~1 GB on disk; reads into memory as a GeoDataFrame)
parcels = gpd.read_file(PARCEL_SHP)

print(f"Total parcels loaded : {len(parcels):,}")
print(f"CRS                  : {parcels.crs}")
print(f"UseCode sample values: {parcels['UseCode'].dropna().unique()[:10]}")


Total parcels loaded : 489,576
CRS                  : EPSG:3857
UseCode sample values: ['1100' '0800' '3600' '0500' '0300' '8300' '4100' '4102' '2500' '3000']



## 1.2 Filter Parcels by Use Code (3xxx / 8xxx / 9xxx)

Retain only parcels whose 4-digit `UseCode` begins with **3** (commercial),
**8** (institutional / public), or **9** (miscellaneous / special).  
The result is saved to `parcels_use_code_filtered.gpkg`.


In [4]:

# Cast UseCode to string and normalise to 4 characters (zero-pad if needed),
# then keep rows whose first character matches one of the target prefixes.
parcels["UseCode_str"] = (
    parcels["UseCode"]
    .astype(str)
    .str.strip()
    .str.zfill(4)
)

mask_use_code = parcels["UseCode_str"].str[0].isin(list(USE_CODE_PREFIXES))
parcels_use_code = parcels[mask_use_code].copy()

print(f"Parcels matching 3xxx/8xxx/9xxx use codes : {len(parcels_use_code):,}")
print(parcels_use_code["UseCode_str"].value_counts().head(15))

# Save to GeoPackage
os.makedirs(OUT_DIR, exist_ok=True)
parcels_use_code.to_file(OUT_USE_CODE, driver="GPKG")
print(f"\nSaved → {OUT_USE_CODE}")


Parcels matching 3xxx/8xxx/9xxx use codes : 15,038
UseCode_str
3100    2361
3200    1961
9400    1475
3000     948
3900     856
8100     813
9300     676
8300     668
3600     642
3300     606
9401     523
9901     517
3700     456
8500     395
8200     222
Name: count, dtype: int64

Saved → ../../data/corridors/parcels/parcels_use_code_filtered.gpkg



## 1.3 Load Business Data and Build GeoDataFrame

Read the DataAxle business registry from the Excel file.  
`Latitude` / `Longitude` are in **WGS 84 (EPSG:4326)**, so we construct point
geometries and then reproject to match the parcel CRS (**EPSG:3857**) before
performing the spatial join.


In [5]:

# Load business registry (only need the coordinate columns for spatial work)
businesses_df = pd.read_excel(BUSINESS_XLSX)

# Drop rows that have no usable coordinates
businesses_df = businesses_df.dropna(subset=["Latitude", "Longitude"])

print(f"Businesses with valid coordinates : {len(businesses_df):,}")

# Build GeoDataFrame — points from lat/lon (WGS 84)
businesses_gdf = gpd.GeoDataFrame(
    businesses_df,
    geometry=gpd.points_from_xy(businesses_df["Longitude"], businesses_df["Latitude"]),
    crs=BUSINESS_CRS,
)

# Reproject to the same CRS as the parcel layer (EPSG:3857)
businesses_gdf = businesses_gdf.to_crs(PARCEL_CRS)

print(f"Business CRS after reprojection   : {businesses_gdf.crs}")


Businesses with valid coordinates : 29,545
Business CRS after reprojection   : EPSG:3857



## 1.4 Filter Parcels That Contain a Business Centroid

Use a spatial join (`predicate="contains"`) to find every parcel polygon that
contains at least one business point.  We then deduplicate by parcel index so
each parcel appears only once regardless of how many businesses it contains.  
The result is saved to `parcels_with_business.gpkg`.


In [6]:

# Spatial join: keep only parcel rows that contain ≥1 business point.
# how="inner" returns one row per (parcel, business) match — we deduplicate after.
joined = gpd.sjoin(
    parcels,                   # left  — polygon layer
    businesses_gdf[["geometry"]],  # right — point layer (geometry only to keep output slim)
    how="inner",
    predicate="contains",
)

# Deduplicate: one row per parcel (a parcel may match many businesses)
parcels_with_biz = parcels.loc[joined.index.unique()].copy()

print(f"Parcels containing ≥1 business centroid : {len(parcels_with_biz):,}")

# Save to GeoPackage
parcels_with_biz.to_file(OUT_BIZ_CONTAIN, driver="GPKG")
print(f"Saved → {OUT_BIZ_CONTAIN}")


Parcels containing ≥1 business centroid : 16,041
Saved → ../../data/corridors/parcels/parcels_with_business.gpkg



## 1.5 Build the Union Layer

Concatenate the two filtered parcel sets and drop duplicate rows (by original
DataFrame index) so that a parcel matching both criteria appears only once.  
The result is saved to `parcels_union.gpkg`.


In [7]:

# Combine the two filtered layers and deduplicate by original row index
parcels_union = (
    pd.concat([parcels_use_code, parcels_with_biz])
    .pipe(gpd.GeoDataFrame, crs=PARCEL_CRS)
    # Reset index to expose it as a column, drop duplicate original indices
    .reset_index(drop=False)
    .drop_duplicates(subset="index")
    .set_index("index")
)

print(f"Parcels in use-code filter   : {len(parcels_use_code):,}")
print(f"Parcels with business        : {len(parcels_with_biz):,}")
print(f"Parcels in union (no dups)   : {len(parcels_union):,}")

# Save to GeoPackage
parcels_union.to_file(OUT_UNION, driver="GPKG")
print(f"\nSaved → {OUT_UNION}")


Parcels in use-code filter   : 15,038
Parcels with business        : 16,041
Parcels in union (no dups)   : 28,236

Saved → ../../data/corridors/parcels/parcels_union.gpkg
